# Homework 09
In this assignment, you will practice writing SQL queries to pull data from a relational database. 

## The congress database
A visual schema of the congress database is also available as a pdf along with this file - we encourage you to start by taking a look to get familiar. This database contains the history of the members of the United States congress through the 115th congress (the data end by 2019) as well as a good deal of voting data from 2015-2016. The visual schema shows each table in the database as a yellow box with the table name at the top and the column names listed below. The arrows showing which keys/identifiers match between different tables for join operations. At the bottom you can see previews of the tables. One in particular, the `cur_members` table, contains data about members of the 115th congress (note - everywhere you see us refer to "current" members below, we are referring to those in the `cur_members` table). Some of these members are still serving as of the 118th congress, which began in January 2023.

## How to use SQL and report your results

You should use Python sqlite3 and Pandas to run your SQL queries in this assignment. We recommend using multiline strings for your queries for readability; remember that multiline strings in Python are enclosed by triple quotes for example,

```
my_multiline_string = """Hi 
There!"""
```

See the worked examples for an example query and it's result.

## Questions

In [1]:
# Run but do not modify the following code
# to import sqlite3 and pandas, and to connect
# to the congress database
import sqlite3
import pandas as pd
import numpy as np
conn = sqlite3.connect("congress")
# If you cannot load the database and see error message like "no such table", try use absolute path like following:
# import sqlite3
# import pandas as pd
# import numpy as np
# import os.path
# BASE_DIR = os.path.dirname(os.path.abspath("congress"))
# db_path = os.path.join(BASE_DIR, "congress")
# conn = sqlite3.connect(db_path)

### Question 1 (4 points)
Who are all of the members of congress from North Carolina (`NC`) in the `cur_members` table? Show all of the information about them from the `cur_members` table. Put your resulting DataFrame in `q1`.

In [2]:
# Start with a SQL query to extract the information
query1 = """
SELECT *
FROM cur_members
WHERE state = 'NC';
"""

# Execute queries using pandas
q1 = pd.read_sql_query(query1, conn)

# Leave this line here so that the table is shown
q1

,id,first_name,last_name,gender,birthday,religion,type,party,state
0,B001135,Richard,Burr,M,1955-11-30,Methodist,sen,Republican,NC
1,B001251,George,Butterfield,M,1947-04-27,Baptist,rep,Democrat,NC
2,F000450,Virginia,Foxx,F,1943-06-29,Roman Catholic,rep,Republican,NC
3,J000255,Walter,Jones,M,1943-02-10,Catholic,rep,Republican,NC
4,M001156,Patrick,McHenry,M,1975-10-22,,rep,Republican,NC
5,P000523,David,Price,M,1940-08-17,Baptist,rep,Democrat,NC
6,H001067,Richard,Hudson,M,1971-11-04,,rep,Republican,NC
7,P000606,Robert,Pittenger,M,1948-08-15,,rep,Republican,NC
8,M001187,Mark,Meadows,M,1959-07-28,,rep,Republican,NC
9,H001065,George,Holding,M,1968-04-17,,rep,Republican,NC


### Question 2 (4 points)
List the five youngest female (`gender = 'F'`) members of the Congress from the `cur_members` table in descending order of `birthday`. Show all information about them from the `cur_members` table. Put your resulting DataFrame in `q2`.

In [3]:
# Start with a SQL query to extract the information
query2 = """
SELECT *
FROM cur_members
WHERE gender = 'F'
ORDER BY birthday DESC
LIMIT 5;
"""

# Execute queries using pandas
q2 = q2 = pd.read_sql_query(query2, conn)


# Leave this line here so that the table is shown
q2

,id,first_name,last_name,gender,birthday,religion,type,party,state
0,S001196,Elise,Stefanik,F,1984-07-02,,rep,Republican,NY
1,G000571,Tulsi,Gabbard,F,1981-04-12,,rep,Democrat,HI
2,H001056,Jaime,Herrera Beutler,F,1978-11-03,,rep,Republican,WA
3,M001202,Stephanie,Murphy,F,1978-09-16,,rep,Democrat,FL
4,B001300,Nanette,Barragán,F,1976-09-15,,rep,Democrat,CA


### Question 3 (4 points)
For the current members of congress, show the average, minimum, and maximum age (in years) of the representatives (i.e., `type = 'rep'`). Show the same for senators (i.e., `type = 'sen'`). Compute ages in years as of the current date (`current_date`). You may compute the age of an individual as simply `current_date - birthday`. The autograder will accept answers computed during any day in the semester.

Note that this is not *exactly* correct due to date rounding error. A more precise calculation of age in years using [sqlite date and time functions](https://sqlite.org/lang_datefunc.html) looks like `(julianday(current_date)-julianday(birthday)) / 365.25`. We will accept solutions using either, you will get very similar results.

Put your resulting DataFrame in `q3` with colums of `type`, `avg`, `min`, and `max`.

In [4]:
# Start with a SQL query to extract the information
query3 = """
SELECT 
    type,
    AVG((julianday('now') - julianday(birthday)) / 365.25) AS avg,
    MIN((julianday('now') - julianday(birthday)) / 365.25) AS min,
    MAX((julianday('now') - julianday(birthday)) / 365.25) AS max
FROM cur_members
GROUP BY type;
"""

# Execute queries using pandas & check for the empty string
q3 = q3 = pd.read_sql_query(query3, conn)



# Leave this line here so that the table is shown
q3

,type,avg,min,max
0,rep,67.405295,41.735281,96.864644
1,sen,71.437567,48.872858,92.763344


### Question 4 (4 points)
Which parties (`party`) have had at least 50 senators (`sen`) past and present? For each, show the `party` and the number of senators (`sen`) they have had. 

Hint: Be sure not to double count persons who served multiple terms in the senate. You can use `COUNT(DISTINCT column)` to get the number of distinct values on given `column`.

Put your resulting DataFrame in `q4` with colums of `party` and `num_senators`.

In [5]:
# Start with a SQL query to extract the information
query4 = """
SELECT 
    party,
    COUNT(DISTINCT person_id) AS num_senators
FROM person_roles
WHERE type = 'sen'
GROUP BY party
HAVING COUNT(DISTINCT person_id) >= 50;
"""

# Execute queries using pandas
q4 = pd.read_sql_query(query4, conn)


# Leave this line here so that the table is shown
q4

,party,num_senators
0,Democrat,842
1,Federalist,74
2,Jackson,68
3,Republican,877
4,Whig,74


### Question 5 (4 points)
Among the past and present members members of congress who cast at least one vote in the `person_votes` table (i.e., the members whose `person_id`s appear at all in the `person_votes` table), which five members cast the fewest votes? Show their `first_name`s, `last_name`s, and the number of votes they cast in the `person_votes` table. Put your resulting DataFrame in `q5` with colums of `first_name`, `last_name`, and `num_votes`.

Note: There is a vote called "Not Voting", you do not need to treat them specially. Or in other words, do not filter them out when counting, count them as a vote.

In [6]:
# Start with a SQL query to extract the information
query5 = """
SELECT 
    p.first_name,
    p.last_name,
    COUNT(pv.vote_id) AS num_votes
FROM persons p
JOIN person_votes pv
    ON p.id = pv.person_id
GROUP BY p.id
ORDER BY num_votes ASC
LIMIT 5;
"""

# Execute queries using pandas
q5 = pd.read_sql_query(query5, conn)


# Leave this line here so that the table is shown
q5

,first_name,last_name,num_votes
0,John,Boehner,17
1,James,Comer,47
2,Dwight,Evans,47
3,Colleen,Hanabusa,47
4,Alan,Nunnelee,51


### Question 6 (4 points)
In this question, we would like to consider the change in the gender representation of the US House of Representatives over time. Find all of the `start_date`s from the `person_roles` table in which at least 40 women began terms as representatives in the house (i.e., `type='rep'`). Show the `start_date`s and the number of women beginning terms on those dates. Order the results by `start_date` from oldest to most recent. Put your resulting DataFrame in `q6` with colums of `start_date` and `house_women`.

In [7]:
# Start with a SQL query to extract the information
query6 = """
SELECT 
    pr.start_date,
    COUNT(*) AS house_women
FROM person_roles pr
JOIN persons p
    ON pr.person_id = p.id
WHERE pr.type = 'rep'
  AND p.gender = 'F'
GROUP BY pr.start_date
HAVING COUNT(*) >= 40
ORDER BY pr.start_date ASC;
"""

# Execute queries using pandas
q6 = pd.read_sql_query(query6, conn)


# Leave this line here so that the table is shown
q6

,start_date,house_women
0,1993-01-05,48
1,1995-01-04,48
2,1997-01-07,53
3,1999-01-06,58
4,2001-01-03,61
5,2003-01-07,62
6,2005-01-04,68
7,2007-01-04,74
8,2009-01-06,78
9,2011-01-05,75


<!-- BEGIN QUESTION -->

# AI Disclosure

Use the **Artificial Intelligence Disclosure (AID) Framework** to explain your use of AI on this assignment. Other headings you can use include:

- *Debugging:* Using AI to help you fix your code so that it works. You should state how you used it for this purpose.

Here are some examples:

*Artificial Intelligence Tools:* ChatGPT v5 via chatgpt.com. *Conceptualization:* I gave chatgpt.com the election data set and asked it for ideas on interesting statistics I could get from the data. *Methodology:* I asked it for help on how to write the code to get the statistic I chose, but I wrote the code myself. *Writing — Review & Editing:* I wrote out my explanation for what the statistic meant, then gave that text and the rubric to chatgpt and asked it to give me feedback on how to update the explanation to conform to the rubric.

*Artificial Intelligence Tools:* ChatGPT v4o via DukeGPT. *Information Collection:* DukeGPT was used to find the function needed to get the index value of the maximum value of a Series and the syntax needed to filter rows in a pandas dataframe using multiple columns. *Debugging:* DukeGPT was used to help me find a bug in my code for Q1 where I copied in the code and error, stated what the code should do, and asked for help.

**Solution:** Artificial Intelligence Tools: ChatGPT v5 chatgpt.com. *Debugging*: I gave my code to chatgpt.com and any errors I found in running my code, and it helped me when I stated what the code should do.

<!-- END QUESTION -->

## Submitting

You should make sure any code that you write to answer the questions is included in this notebook. You are **required** to go to the Kernel option and choose **"Restart & Run All"**  before submission. Double check that your entire notebook runs correctly and generates the expected output. Finally, make sure to save your work (timestamp at the top tells you the last checkpoint and whether there are unsaved changes). When you finish, submit your assignment at [Gradescope](http://gradescope.com/ "‌"). **Submissions not prepared correctly as above will lose points.**